In [ ]:
import pandas as pd
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel

In [ ]:
DEVICE = torch.device("cuda:0")

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.add_special_tokens({"pad_token": "<|endoftext|>"})

model = GPT2LMHeadModel.from_pretrained('gpt2')
# model.set_attn_implementation("eager")
model.to(DEVICE)
if DEVICE.type == "cuda":
    model.set_attn_implementation("flash_attention_2")

In [ ]:
model.config.output_attentions = True

In [ ]:
# text = "Do androids dream of electric memes?"
# encoded_input = tokenizer(text, return_tensors='pt').to(DEVICE)
# output = model(**encoded_input)

In [ ]:
sft_dataset_dedup_train = pd.read_parquet("datasets/sft/train_dedup.parquet")
sft_dataset_dedup_val = pd.read_parquet("datasets/sft/val_dedup.parquet")
sft_dataset_dedup_test = pd.read_parquet("datasets/sft/test_dedup.parquet")

In [ ]:
sft_dataset_dedup_train["sft_prompt"] = "You are given a text. Your task is to write a concise summary for this text. Text: " + sft_dataset_dedup_train["post"] + " Summary: "
sft_dataset_dedup_val["sft_prompt"] = "You are given a text. Your task is to write a concise summary for this text. Text: " + sft_dataset_dedup_val["post"] + " Summary: "
sft_dataset_dedup_test["sft_prompt"] = "You are given a text. Your task is to write a concise summary for this text. Text: " + sft_dataset_dedup_test["post"] + " Summary: "

In [ ]:
sorted_vocab = sorted(tokenizer.get_vocab().items(), key=lambda x: x[1])

In [ ]:
# sorted_vocab[50257]

In [ ]:
test_input = {
    "text": list(sft_dataset_dedup_train["sft_prompt"][:2]),
    "text_target": list(sft_dataset_dedup_train["summary"][:2])
}

In [ ]:
test_input_tensors = tokenizer(test_input["text"], padding=True, truncation=True, return_tensors="pt", add_special_tokens=True, padding_side="left")
test_label_tensors = tokenizer(test_input["text_target"], padding=True, truncation=True, return_tensors="pt", add_special_tokens=True, padding_side="right")

In [ ]:
test_llm_input = {
    "input_ids": torch.cat((test_input_tensors["input_ids"], test_label_tensors["input_ids"]), dim=1).to(DEVICE),
    "attention_mask": torch.cat((test_input_tensors["attention_mask"], test_label_tensors["attention_mask"]), dim=1).to(DEVICE)
}

In [ ]:
res = model(
    **test_llm_input
)

In [ ]:
res.attentions[-1]